# Rebuild `index_v2` — the missing piece

The v2 model (`b5_real_finetuned_v2`) exists in Drive, but the retrieval index
built from it was never saved and was lost when the Colab session ended.

This notebook rebuilds it and saves it to Drive so it never has to happen again.

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save.

Run every cell top to bottom.


## 1. Confirm GPU

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU ONLY - go enable the T4")
assert torch.cuda.is_available(), "Enable the T4 GPU first: Runtime -> Change runtime type"

## 2. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ROOT   = "/content/drive/MyDrive"
P1     = f"{ROOT}/Phase1_Project/MemberB_B4_B6_output"
P1FIX  = f"{ROOT}/Phase1_Project/data_fix_output"
GV2    = f"{ROOT}/Phase3_Project/guardrail_output_v2"

for p in [P1, P1FIX, GV2]:
    print(("OK   " if __import__("os").path.isdir(p) else "MISS "), p)

## 3. Get the code

Two sources, merged. GitHub has 27 files; Drive's `guardrail_output_v2/src`
has 22 including `augment_training_with_ayatec.py`, which GitHub is missing.
We take the union — Drive wins on conflicts, since it's what actually ran.


In [ ]:
import os, shutil, subprocess

PROJECT = "/content/QuranicRAG"
shutil.rmtree(PROJECT, ignore_errors=True)
os.makedirs(f"{PROJECT}/src", exist_ok=True)
os.makedirs(f"{PROJECT}/quranNLP/shared/data", exist_ok=True)
os.makedirs(f"{PROJECT}/data", exist_ok=True)
os.chdir(PROJECT)

# Public repo - no upload needed.
subprocess.run(["git","clone","--depth","1",
    "https://github.com/Laiba-Noor/quranic-rag-hallucination-free.git",
    "/content/_repo"], check=True)

for f in os.listdir("/content/_repo/src"):
    if f.endswith(".py"):
        shutil.copy(f"/content/_repo/src/{f}", f"src/{f}")
n_github = len(os.listdir("src"))

overlaid = []
for f in sorted(os.listdir(f"{GV2}/src")):
    if f.endswith(".py"):
        shutil.copy(f"{GV2}/src/{f}", f"src/{f}")
        overlaid.append(f)

print(f"from GitHub: {n_github} files")
print(f"overlaid from Drive: {len(overlaid)} files")
print(f"total in src/: {len(os.listdir('src'))}")
print("\naugment_training_with_ayatec.py present:",
      os.path.exists("src/augment_training_with_ayatec.py"))

## 4. Install dependencies

In [ ]:
!pip install -q sentence-transformers hnswlib pandas rapidfuzz pyarabic networkx
print("done")

## 5. Stage the corpus

`b6` reads `quranNLP/shared/data/final_cross_reference_index.csv` — that exact
path is hardcoded. Drive has it; we copy rather than upload 90 MB by hand.


In [ ]:
import os, shutil

CSV_DST = "quranNLP/shared/data/final_cross_reference_index.csv"
candidates = [
    f"{P1FIX}/shared_data/final_cross_reference_index.csv",
    "/content/_repo/Data/final_cross_reference_index.csv",
]
for c in candidates:
    if os.path.exists(c):
        print(f"copying from {c}  ({os.path.getsize(c)/1e6:.1f} MB)")
        shutil.copy(c, CSV_DST)
        break
else:
    raise SystemExit("final_cross_reference_index.csv not found anywhere")

# Supporting data the later phases want.
for src_dir, names in [
    (f"{P1FIX}/shared_data", ["clean_verses.csv", "tafsir_verse_mappings.csv"]),
    (f"{ROOT}/Phase2_Project/Roma_output/data", ["qrcd_flat.json", "sufficiency_labels.json"]),
    ("/content/_repo/Data", ["ayatec_records.json", "squad_v2_sample.json"]),
]:
    for n in names:
        s = os.path.join(src_dir, n)
        if os.path.exists(s):
            shutil.copy(s, f"quranNLP/shared/data/{n}")
            print("  staged", n)

print("\nquranNLP/shared/data/:")
for f in sorted(os.listdir("quranNLP/shared/data")):
    print(f"   {f}  ({os.path.getsize('quranNLP/shared/data/'+f)/1e6:.1f} MB)")

## 6. Pull the v2 model down from Drive

Copying locally first — building the index straight off Drive is painfully slow.

In [ ]:
import shutil, os, time
t = time.time()
shutil.copytree(f"{GV2}/b5_real_finetuned_v2", "b5_real_finetuned_v2", dirs_exist_ok=True)
print(f"copied in {time.time()-t:.0f}s")
print(sorted(os.listdir("b5_real_finetuned_v2")))

## 7. Build the index

This is the actual work — encoding every verse and every direct tafsir passage
with the v2 model, then building the HNSW graph. Expect 15–40 minutes on a T4.
Don't let the tab sleep.


In [ ]:
%cd /content/QuranicRAG
!python src/b6_build_index_and_retrieval_api.py --model-path ./b5_real_finetuned_v2

## 8. Check it built

In [ ]:
import os
for f in ["index/verses.hnsw", "index/entries.pkl"]:
    print(("OK   " if os.path.exists(f) else "MISS "), f,
          f"({os.path.getsize(f)/1e6:.1f} MB)" if os.path.exists(f) else "")

## 9. Save it to Drive — **do not skip this**

This is the exact step that was missed last time. The index lives only in the
Colab VM until this cell runs; when the session ends it is gone.


In [ ]:
import shutil, os
DEST = f"{GV2}/index_v2"
shutil.copytree("index", DEST, dirs_exist_ok=True)
print("saved to:", DEST)
for f in sorted(os.listdir(DEST)):
    print(f"   {f}  ({os.path.getsize(os.path.join(DEST,f))/1e6:.1f} MB)")

## 10. Smoke test — does retrieval actually work?

In [ ]:
%cd /content/QuranicRAG
import sys; sys.path.insert(0, "src")
from sentence_transformers import SentenceTransformer
from b6_build_index_and_retrieval_api import load_index, RetrievalAPI

model = SentenceTransformer("./b5_real_finetuned_v2")
index, entries = load_index(dim=model.get_sentence_embedding_dimension(), out_dir="index")
api = RetrievalAPI(model, index, entries)
print(f"{len(entries)} entries indexed\n")

for q in ["ما فوائد الصبر في القرآن", "من هو النبي الذي ابتلعه الحوت"]:
    print("=" * 60)
    print("QUERY:", q)
    for r in api.retrieve(q, top_k=3):
        print(f"  [{r['verse_key']}] {r['source_type']:<7} sim={r.get('score', 0):.3f}")
        print(f"      {r['text'][:90]}")

---

## When this finishes

You'll have, saved permanently in Drive:

- `b5_real_finetuned_v2` — the improved model (already there)
- `index_v2` — its retrieval index (new)

That's a complete, working Phase 1–3 system you can run any time.
Next up is Phase 4: the evaluation harness.
